<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Prod/onnxruntime/2_onnxrt_paths.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Installation

ONNX Runtime has many versions, for example:

- onnxruntime
- onnxruntime-gpu
- onnxruntime-openvino
...

Some providers, like OpenVINO, have their own specific packages, whereas TensorRT does not. However, during inference, you should still use an import like this:

> import onnxruntime

regardless of which package you installed. This means **you cannot install different ONNX Runtime packages in the same environment**, even if they have different names.


In [ ]:
#!pip install onnxruntime-gpu tensorrt

# Run Attempt

We installed `onnxruntime-gpu` and `tensorrt`, and we already have an ONNX model on disk:

`model_repo/resnet18_onnx/1/model.onnx`

So the first expectation is that ONNX Runtime should be able to run the model with `TensorrtExecutionProvider`.


In [1]:
import numpy as np
import onnxruntime as ort
#set_tensorrt_ld_library_path()

sess = ort.InferenceSession("model/model.onnx",
                            providers=["TensorrtExecutionProvider"])
dummy = {"INPUT_BATCH_OF_IMAGES": np.random.rand(1, 3, 224, 224).astype(np.float32)}
out = sess.run(None, dummy)

print(f"providers={sess.get_providers()}  output={out[0].shape}")

providers=['TensorrtExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)


But the run fails.

Why does this happen? Maybe the required providers are not installed?
Let's check.


In [2]:
print("Available providers:", ort.get_available_providers())

Available providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


A listed provider is not enough. ONNX Runtime must also be able to load all native TensorRT, CUDA, and cuDNN libraries required by that provider.

ONNX Runtime ships a library named `libonnxruntime_providers_tensorrt.so`.

That library depends on the actual TensorRT libraries:

- `libnvinfer.so.*`
- `libnvonnxparser.so.*`

It also depends on CUDA and cuDNN native libraries:

- `libcudart.so.*`
- `libcudnn.so.*`

These paths are not hardcoded to your Python environment. The [dynamic loader](https://man7.org/linux/man-pages/man8/ld-linux.so.8.html) resolves them through system library paths and `LD_LIBRARY_PATH`.

In this environment, TensorRT and cuDNN were installed inside Python `site-packages`, so ONNX Runtime could not see them until those directories were added to `LD_LIBRARY_PATH` before the kernel started.

First, check what the ONNX Runtime TensorRT provider can currently resolve:


In [3]:
import site, subprocess
site_packages = site.getsitepackages()[0]
target = f"{site_packages}/onnxruntime/capi/libonnxruntime_providers_tensorrt.so"
ldd_out = subprocess.check_output(["ldd", target], text=True)

groups = {"TensorRT": ["libnvinfer.so", "libnvonnxparser.so"], "CUDA": ["libcudart.so"], "cuDNN": ["libcudnn.so"]}

for label, prefixes in groups.items():
    print(f"{label}:")
    for prefix in prefixes:
        line = next((l for l in ldd_out.splitlines() if prefix in l), None)
        name = line.split("=>")[0].strip() if line and "=>" in line else prefix
        path = line.split("=>")[1].split("(")[0].strip() if line and "=>" in line else "not found"
        print(f"\t{name} | {path}")


TensorRT:
	libnvinfer.so.10 | not found
	libnvonnxparser.so.10 | not found
CUDA:
	libcudart.so.12 | /usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12
cuDNN:
	libcudnn.so.9 | not found


Where are the TensorRT libraries installed?


In [7]:
from pathlib import Path
import site

site_packages = Path(site.getsitepackages()[0])
wanted_prefixes = groups["TensorRT"] + groups["cuDNN"]
add_to_ld_library_path = set()
for prefix in wanted_prefixes:
    # Find all matching files and take the first one found, or 'not found'
    match = next(site_packages.rglob(f"{prefix}*"), "not found")
    print(f"{prefix} | {match}")
    if match != "not found":
        add_to_ld_library_path.add(str(match.parent))


libnvinfer.so | /home/gan/Code/WB/cv/Prod/onnxruntime/.venv/lib/python3.10/site-packages/tensorrt_libs/libnvinfer.so.10
libnvonnxparser.so | /home/gan/Code/WB/cv/Prod/onnxruntime/.venv/lib/python3.10/site-packages/tensorrt_libs/libnvonnxparser.so.10
libcudnn.so | /home/gan/Code/WB/cv/Prod/onnxruntime/.venv/lib/python3.10/site-packages/nvidia/cudnn/lib/libcudnn.so.9


## How to Fix

1. **Install missed libraries**
In this case we have to use cuDNN version compatible to CUDA used by Onnxruntime version (`/usr/local/cuda-12.4`)

In [5]:
!pip install nvidia-cudnn-cu12

  Using cached nvidia_cudnn_cu12-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl (657.9 MB)



2. **Add the paths** to the missing files to `LD_LIBRARY_PATH`.

For TensorRT EP, we need:
- `tensorrt_libs` from Python `site-packages`
- `nvidia/cudnn/lib` from Python `site-packages`
- the system CUDA runtime directory that contains `libcudart.so`

On this machine, the CUDA path is already visible, so the missing pieces were `tensorrt_libs` and `nvidia/cudnn/lib`.

In VS Code notebooks, setting `LD_LIBRARY_PATH` inside a running cell is too late in this case. The practical fix is to put it into the `.env` file in the workspace root, then restart the kernel.

Example:

```env
LD_LIBRARY_PATH=path/to/site-packages/tensorrt_libs:path/to/site-packages/nvidia/cudnn/lib
```


For this machine add to `LD_LIBRARY_PATH`

In [8]:
":".join(add_to_ld_library_path)

'/home/gan/Code/WB/cv/Prod/onnxruntime/.venv/lib/python3.10/site-packages/nvidia/cudnn/lib:/home/gan/Code/WB/cv/Prod/onnxruntime/.venv/lib/python3.10/site-packages/tensorrt_libs'

# Conclusion

Installing TensorRT into a Python virtual environment for use with ONNX Runtime is a bad idea.
It is better to install it system-wide:

```bash
sudo apt-get install tensorrt
```
